---
## 0. Cài đặt thư viện & Import

In [1]:
# Cài đặt thư viện cần thiết (chạy một lần)
import subprocess, sys

packages = ["pyspark", "matplotlib", "seaborn", "pandas", "numpy", "scikit-learn"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Đã cài đặt tất cả thư viện!")

Đã cài đặt tất cả thư viện!


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# ── PySpark ──────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

# ── MLlib — Feature Engineering ──────────────────────────────────
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, Imputer
)

# ── MLlib — Models ───────────────────────────────────────────────
from pyspark.ml.classification import (
    LinearSVC,
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier
)

# ── MLlib — Evaluation ───────────────────────────────────────────
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# ── MLlib — Tuning ───────────────────────────────────────────────
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# ── Matplotlib style ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#2d3561',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#a0aab4',
    'ytick.color':      '#a0aab4',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2d3561',
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})
PALETTE = ['#6c63ff', '#ff6584', '#43d9ad', '#ffd166', '#ef8c4a']

print("Import hoàn tất!")

---
## 1. Khởi động Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("SVM_ECommerce_Classification_QuynhTrang")
    .master("local[*]")                          # Dùng tất cả CPU core
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.default.parallelism", "50")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version : {spark.version}")
print(f"   App name      : {spark.sparkContext.appName}")
print(f"   Master        : {spark.sparkContext.master}")
print(f"   Spark UI       : http://localhost:4040")

---
## 2. Đọc Dữ Liệu

> **Ưu tiên đọc từ HDFS.** Nếu HDFS chưa chạy, tự động fallback sang file local.

In [ ]:
# ── Cấu hình đường dẫn ───────────────────────────────────────────
HDFS_PATH  = "hdfs://namenode:9000/ecom/Pakistan_Largest_Ecommerce_Dataset.csv"
LOCAL_PATH = "/home/jovyan/work/Pakistan Largest Ecommerce Dataset.csv"

def load_dataset(spark, hdfs_path, local_path):
    """Thử đọc từ HDFS, fallback sang local nếu lỗi."""
    common_opts = {
        "header": "true",
        "inferSchema": "true",
        "nullValue": "#N/A",
        "multiLine": "false",
        "encoding": "UTF-8",
    }
    try:
        df = spark.read.options(**common_opts).csv(hdfs_path)
        print(f"Đọc từ HDFS: {hdfs_path}")
        return df, "HDFS"
    except Exception as e:
        print(f"HDFS không khả dụng ({e.__class__.__name__}). Đọc file local...")
        df = spark.read.options(**common_opts).csv(local_path)
        print(f"Đọc từ local: {local_path}")
        return df, "LOCAL"

raw_df, source = load_dataset(spark, HDFS_PATH, LOCAL_PATH)

print(f"\n Số dòng : {raw_df.count():,}")
print(f"   Số cột  : {len(raw_df.columns)}")
print(f"   Nguồn   : {source}")

In [ ]:
# Xem 5 dòng đầu
raw_df.limit(5).toPandas()

In [ ]:
# Schema
raw_df.printSchema()

---
## 3. Khám Phá Dữ Liệu (EDA)

In [ ]:
# ── Thống kê mô tả ───────────────────────────────────────────────
num_cols = ["price", "qty_ordered", "grand_total", "discount_amount"]
raw_df.select(num_cols).describe().toPandas().set_index("summary")

In [ ]:
# ── Phân phối trạng thái đơn hàng ───────────────────────────────
status_dist = (
    raw_df.groupBy("status")
          .count()
          .orderBy(F.col("count").desc())
          .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Phân Phối Trạng Thái Đơn Hàng', fontsize=16, fontweight='bold', color='white', y=1.02)

# Bar chart
colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(status_dist)))
bars = axes[0].barh(status_dist['status'], status_dist['count'], color=colors, edgecolor='none')
axes[0].set_xlabel('Số đơn hàng', labelpad=10)
axes[0].set_title('Số lượng theo trạng thái', fontweight='bold')
axes[0].invert_yaxis()
for bar, val in zip(bars, status_dist['count']):
    axes[0].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9, color='#a0aab4')

# Pie chart
top6 = status_dist.head(6)
axes[1].pie(top6['count'], labels=top6['status'], autopct='%1.1f%%',
            colors=PALETTE * 2, startangle=140,
            wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2},
            textprops={'color': 'white', 'fontsize': 9})
axes[1].set_title('Tỷ lệ top 6 trạng thái', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_status_dist.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(status_dist.to_string(index=False))

In [ ]:
# ── Phân phối các đặc trưng số ───────────────────────────────────
num_pdf = raw_df.select(num_cols).dropna().limit(50000).toPandas()
# Clamp outliers
for c in num_cols:
    q99 = num_pdf[c].quantile(0.99)
    num_pdf[c] = num_pdf[c].clip(upper=q99)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Phân Phối Các Đặc Trưng Số (Cắt Outlier 99%)', fontsize=14, fontweight='bold', color='white')

titles = ['Giá sản phẩm (price)', 'Số lượng (qty_ordered)', 'Tổng tiền (grand_total)', 'Giảm giá (discount_amount)']
for ax, col, title, color in zip(axes, num_cols, titles, PALETTE):
    ax.hist(num_pdf[col].dropna(), bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Số đơn hàng')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_feature_dist.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── Top danh mục và phương thức thanh toán ───────────────────────
cat_dist = (
    raw_df.groupBy("category_name_1").count()
          .orderBy(F.col("count").desc()).limit(10).toPandas()
)
pay_dist = (
    raw_df.groupBy("payment_method").count()
          .orderBy(F.col("count").desc()).limit(8).toPandas()
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Phân Phối Danh Mục & Phương Thức Thanh Toán', fontsize=14, fontweight='bold', color='white')

# Top categories
c1 = plt.cm.viridis(np.linspace(0.3, 0.9, len(cat_dist)))
ax1.barh(cat_dist['category_name_1'], cat_dist['count'], color=c1[::-1], edgecolor='none')
ax1.set_xlabel('Số đơn hàng')
ax1.set_title('Top 10 Danh Mục Sản Phẩm', fontweight='bold')
ax1.invert_yaxis()

# Payment method
c2 = plt.cm.plasma(np.linspace(0.2, 0.8, len(pay_dist)))
ax2.bar(pay_dist['payment_method'], pay_dist['count'], color=c2, edgecolor='none')
ax2.set_xlabel('Phương thức thanh toán')
ax2.set_ylabel('Số đơn hàng')
ax2.set_title('Phân Phối Phương Thức Thanh Toán', fontweight='bold')
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('eda_cat_pay.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## 4. Tiền Xử Lý Dữ Liệu

In [ ]:
# ── Danh sách trạng thái "thành công" ────────────────────────────
SUCCESSFUL_STATUSES = [
    'complete', 'closed',
    'order_refunded',          # Đã xử lý → tính là hoàn thành
]

def preprocess(df):
    # 1. Chuẩn hóa tên cột (bỏ khoảng trắng, viết thường)
    for old_col in df.columns:
        new_col = old_col.strip().replace(' ', '_').lower()
        df = df.withColumnRenamed(old_col, new_col)

    # 2. Xóa cột không cần thiết (unnamed, _c21..._c25)
    drop_pattern = [c for c in df.columns
                    if c.startswith('_c') or 'unnamed' in c.lower() or c == 'mv']
    df = df.drop(*drop_pattern)

    # 3. Ép kiểu số
    cast_map = {
        'price':           DoubleType(),
        'qty_ordered':     DoubleType(),
        'grand_total':     DoubleType(),
        'discount_amount': DoubleType(),
    }
    for col_name, dtype in cast_map.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(dtype))

    # 4. Tạo nhãn nhị phân
    df = df.withColumn(
        'label',
        F.when(F.lower(F.col('status')).isin(SUCCESSFUL_STATUSES), 1.0).otherwise(0.0)
    )

    # 5. Tạo đặc trưng bổ sung
    df = (
        df
        .withColumn('discount_rate',
                    F.when(F.col('grand_total') > 0,
                           F.col('discount_amount') / F.col('grand_total'))
                     .otherwise(0.0))
        .withColumn('price_per_unit',
                    F.when(F.col('qty_ordered') > 0,
                           F.col('price') / F.col('qty_ordered'))
                     .otherwise(F.col('price')))
        .withColumn('log_grand_total',
                    F.log1p(F.col('grand_total')))
    )

    # 6. Lọc dòng null ở các cột quan trọng
    key_cols = ['price', 'qty_ordered', 'grand_total',
                'discount_amount', 'status', 'label']
    key_cols_exist = [c for c in key_cols if c in df.columns]
    df = df.dropna(subset=key_cols_exist)

    # 7. Lọc giá trị âm phi lý
    df = df.filter((F.col('price') >= 0) & (F.col('grand_total') >= 0))

    return df

clean_df = preprocess(raw_df)

total = clean_df.count()
pos   = clean_df.filter(F.col('label') == 1).count()
neg   = total - pos

print(f" Sau tiền xử lý:")
print(f"   Tổng dòng         : {total:,}")
print(f"   Đơn thành công (1): {pos:,}  ({pos/total*100:.1f}%)")
print(f"   Đơn thất bại  (0) : {neg:,}  ({neg/total*100:.1f}%)")
print(f"   Tỷ lệ imbalance   : 1:{neg/pos:.1f}")

In [ ]:
# ── Trực quan hóa cân bằng nhãn ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
fig.patch.set_facecolor('#0f1117')

labels_bar = ['Thất bại (0)', 'Thành công (1)']
values_bar = [neg, pos]
bars = ax.bar(labels_bar, values_bar,
              color=['#ff6584', '#43d9ad'],
              edgecolor='none', width=0.5)

for bar, val in zip(bars, values_bar):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.005,
            f'{val:,}\n({val/total*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='white')

ax.set_title('Phân Phối Nhãn (Label Distribution)', fontsize=13, fontweight='bold')
ax.set_ylabel('Số đơn hàng')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(values_bar) * 1.15)

plt.tight_layout()
plt.savefig('label_dist.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── Kiểm tra null sau tiền xử lý ─────────────────────────────────
feature_cols = ['price', 'qty_ordered', 'grand_total', 'discount_amount',
                'discount_rate', 'price_per_unit', 'log_grand_total',
                'payment_method', 'category_name_1', 'label']

null_counts = {
    col: clean_df.filter(F.col(col).isNull()).count()
    for col in feature_cols if col in clean_df.columns
}

null_df = pd.DataFrame(list(null_counts.items()), columns=['Cột', 'Null count'])
null_df['Tỷ lệ (%)'] = (null_df['Null count'] / total * 100).round(2)
print("Kiểm tra giá trị null:")
null_df

---
## 5. Xử Lý Mất Cân Bằng Nhãn (Class Imbalance)

> Dataset thường bị mất cân bằng (nhiều đơn thất bại hơn thành công). Sử dụng **Class Weight** trong LinearSVC.

In [ ]:
# ── Tính trọng số lớp (Class Weight) ────────────────────────────
# w_neg = total / (2 * neg),  w_pos = total / (2 * pos)
w_neg = total / (2.0 * neg)
w_pos = total / (2.0 * pos)

clean_df = clean_df.withColumn(
    'class_weight',
    F.when(F.col('label') == 1.0, w_pos).otherwise(w_neg)
)

print(f"Trọng số lớp 0 (thất bại) : {w_neg:.4f}")
print(f"Trọng số lớp 1 (thành công): {w_pos:.4f}")

---
## 6. Feature Engineering & Pipeline MLlib

In [ ]:
# ── Xác định cột đặc trưng ───────────────────────────────────────
NUM_COLS = [
    'price', 'qty_ordered', 'grand_total', 'discount_amount',
    'discount_rate', 'price_per_unit', 'log_grand_total'
]
CAT_COLS = ['payment_method', 'category_name_1']

# Lọc cột thực sự có trong dataset
NUM_COLS = [c for c in NUM_COLS if c in clean_df.columns]
CAT_COLS = [c for c in CAT_COLS if c in clean_df.columns]

print(f"Đặc trưng số ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Đặc trưng cat ({len(CAT_COLS)}): {CAT_COLS}")

In [ ]:
# ── Điền giá trị null cho cột số ─────────────────────────────────
imputer = Imputer(
    inputCols=NUM_COLS,
    outputCols=[c + '_imp' for c in NUM_COLS],
    strategy='median'
)
NUM_IMP = [c + '_imp' for c in NUM_COLS]

# ── StringIndexer: mã hóa cột categorical ─────────────────────────
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + '_idx',
        handleInvalid='keep'
    )
    for col in CAT_COLS
]
CAT_IDX = [c + '_idx' for c in CAT_COLS]

# ── OneHotEncoder ─────────────────────────────────────────────────
encoder = OneHotEncoder(
    inputCols=CAT_IDX,
    outputCols=[c + '_ohe' for c in CAT_COLS],
    dropLast=True
)
CAT_OHE = [c + '_ohe' for c in CAT_COLS]

# ── VectorAssembler ───────────────────────────────────────────────
assembler = VectorAssembler(
    inputCols=NUM_IMP + CAT_OHE,
    outputCol='features_raw',
    handleInvalid='skip'
)

# ── StandardScaler ────────────────────────────────────────────────
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=False,     # Sparse vector → không dùng withMean=True
    withStd=True
)

print("Các bước feature engineering đã được định nghĩa!")
print(f"   Tổng đặc trưng đầu vào (số + encoded): {len(NUM_IMP)} + {len(CAT_OHE)}")

---
## 7. Chia Tập Train / Test

In [ ]:
# ── Chọn cột cần thiết & stratified split ────────────────────────
work_cols = NUM_COLS + CAT_COLS + ['label', 'class_weight']
work_cols = [c for c in work_cols if c in clean_df.columns]
ml_df = clean_df.select(work_cols).cache()

# Chia 80/20 ngẫu nhiên
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

# Cache để tăng tốc
train_df.cache()
test_df.cache()

train_cnt = train_df.count()
test_cnt  = test_df.count()

print(f"Chia tập dữ liệu (seed=42):")
print(f"   Train: {train_cnt:,} dòng ({train_cnt/(train_cnt+test_cnt)*100:.1f}%)")
print(f"   Test : {test_cnt:,}  dòng ({test_cnt/(train_cnt+test_cnt)*100:.1f}%)")

# Kiểm tra tỷ lệ nhãn trong train/test
for name, subset in [("Train", train_df), ("Test", test_df)]:
    p = subset.filter(F.col('label') == 1).count()
    n = subset.filter(F.col('label') == 0).count()
    total_s = p + n
    print(f"   {name} → positive: {p/total_s*100:.1f}%, negative: {n/total_s*100:.1f}%")

---
## 8. Huấn Luyện Mô Hình SVM (LinearSVC)

In [ ]:
# ── Định nghĩa mô hình SVM ────────────────────────────────────────
svm = LinearSVC(
    featuresCol='features',
    labelCol='label',
    weightCol='class_weight',
    maxIter=100,
    regParam=0.01,          # λ (L2 regularization)
    tol=1e-6,
    standardization=False,  # Đã StandardScaler ở trên
    aggregationDepth=2,
    threshold=0.0           # Decision boundary
)

# ── Xây dựng Pipeline hoàn chỉnh ─────────────────────────────────
svm_pipeline = Pipeline(stages=[
    imputer,
    *indexers,
    encoder,
    assembler,
    scaler,
    svm
])

print("Pipeline SVM đã được tạo:")
for i, stage in enumerate(svm_pipeline.getStages()):
    print(f"   [{i+1}] {stage.__class__.__name__}")

In [ ]:
# ── Huấn luyện ───────────────────────────────────────────────────
import time

print("Đang huấn luyện mô hình SVM...")
t0 = time.time()

svm_model = svm_pipeline.fit(train_df)

train_time = time.time() - t0
print(f"Huấn luyện xong! Thời gian: {train_time:.1f} giây")

In [ ]:
# ── Hệ số mô hình ────────────────────────────────────────────────
lsvc_model = svm_model.stages[-1]   # LinearSVCModel
print(f"Hệ số điều chỉnh (intercept): {lsvc_model.intercept:.6f}")
print(f"Số đặc trưng               : {len(lsvc_model.coefficients)}")
print(f"Số vòng lặp thực tế        : {lsvc_model.summary.totalIterations if hasattr(lsvc_model, 'summary') else 'N/A'}")

---
## 9. Đánh Giá Mô Hình SVM

In [ ]:
# ── Dự đoán ──────────────────────────────────────────────────────
svm_pred = svm_model.transform(test_df)

# ── Các evaluator ────────────────────────────────────────────────
bin_eval   = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
pr_eval    = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderPR')
mc_acc     = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy')
mc_f1      = MulticlassClassificationEvaluator(labelCol='label', metricName='f1')
mc_prec    = MulticlassClassificationEvaluator(labelCol='label', metricName='weightedPrecision')
mc_rec     = MulticlassClassificationEvaluator(labelCol='label', metricName='weightedRecall')

# ── Tính metrics ─────────────────────────────────────────────────
svm_metrics = {
    'AUC-ROC'           : bin_eval.evaluate(svm_pred),
    'AUC-PR'            : pr_eval.evaluate(svm_pred),
    'Accuracy'          : mc_acc.evaluate(svm_pred),
    'F1-Score'          : mc_f1.evaluate(svm_pred),
    'Precision'         : mc_prec.evaluate(svm_pred),
    'Recall'            : mc_rec.evaluate(svm_pred),
    'Training Time (s)' : train_time,
}

print("\n Kết quả đánh giá mô hình LinearSVC:")
print("-" * 35)
for k, v in svm_metrics.items():
    if 'Time' in k:
        print(f"  {k:<22}: {v:.1f}s")
    else:
        print(f"  {k:<22}: {v:.4f}")

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────
pred_pdf = svm_pred.select('label', 'prediction').toPandas()
cm = confusion_matrix(pred_pdf['label'], pred_pdf['prediction'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Confusion Matrix — LinearSVC', fontsize=14, fontweight='bold', color='white')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['Thất bại (0)', 'Thành công (1)'],
            yticklabels=['Thất bại (0)', 'Thành công (1)'],
            ax=axes[0], linewidths=0.5, cbar=True)
axes[0].set_title('Số lượng tuyệt đối', fontweight='bold')
axes[0].set_ylabel('Thực tế', fontsize=11)
axes[0].set_xlabel('Dự đoán', fontsize=11)

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdPu',
            xticklabels=['Thất bại (0)', 'Thành công (1)'],
            yticklabels=['Thất bại (0)', 'Thành công (1)'],
            ax=axes[1], linewidths=0.5, cbar=True)
axes[1].set_title('Tỷ lệ phần trăm', fontweight='bold')
axes[1].set_ylabel('Thực tế', fontsize=11)
axes[1].set_xlabel('Dự đoán', fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrix_svm.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

# Classification Report
print("\n Classification Report:")
print(classification_report(
    pred_pdf['label'], pred_pdf['prediction'],
    target_names=['Thất bại (0)', 'Thành công (1)']
))

---
## 10. Tinh Chỉnh Siêu Tham Số (Hyperparameter Tuning)

> Dùng **CrossValidator** (3-fold) để tìm `regParam` tối ưu cho LinearSVC.

In [ ]:
# ── Lưới tham số ─────────────────────────────────────────────────
lsvc_stage = svm_pipeline.getStages()[-1]  # LinearSVC stage

param_grid = (
    ParamGridBuilder()
    .addGrid(lsvc_stage.regParam,  [0.001, 0.01, 0.1])
    .addGrid(lsvc_stage.maxIter,   [50, 100])
    .build()
)

evaluator_cv = BinaryClassificationEvaluator(
    labelCol='label',
    metricName='areaUnderROC'
)

cv = CrossValidator(
    estimator=svm_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_cv,
    numFolds=3,
    seed=42,
    parallelism=2
)

print(f" Lưới tham số: {len(param_grid)} tổ hợp × 3 folds = {len(param_grid)*3} lần fit")
print(" Đang chạy Cross-Validation... (có thể mất vài phút)")

t1 = time.time()
cv_model = cv.fit(train_df)
cv_time  = time.time() - t1

print(f" Cross-Validation xong! Thời gian: {cv_time:.1f} giây")

In [ ]:
# ── Mô hình tốt nhất ─────────────────────────────────────────────
best_model = cv_model.bestModel
best_lsvc  = best_model.stages[-1]

print(f" Tham số tốt nhất:")
print(f"   regParam : {best_lsvc.getRegParam()}")
print(f"   maxIter  : {best_lsvc.getMaxIter()}")

# Đánh giá mô hình tốt nhất
best_pred = best_model.transform(test_df)
best_auc  = evaluator_cv.evaluate(best_pred)
best_acc  = mc_acc.evaluate(best_pred)
best_f1   = mc_f1.evaluate(best_pred)

print(f"\n Kết quả mô hình SVM tốt nhất (sau CV):")
print(f"   AUC-ROC  : {best_auc:.4f}")
print(f"   Accuracy : {best_acc:.4f}")
print(f"   F1-Score : {best_f1:.4f}")

In [ ]:
# ── Biểu đồ CV scores theo regParam ──────────────────────────────
avg_metrics = cv_model.avgMetrics
param_labels = []
for pm in param_grid:
    reg = pm[lsvc_stage.regParam]
    itr = pm[lsvc_stage.maxIter]
    param_labels.append(f"reg={reg}\niter={itr}")

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0f1117')

x = np.arange(len(avg_metrics))
bars = ax.bar(x, avg_metrics, color=PALETTE[0], alpha=0.85, edgecolor='none', width=0.6)
ax.set_xticks(x)
ax.set_xticklabels(param_labels, fontsize=9)
ax.set_ylabel('AUC-ROC (CV Mean)')
ax.set_title('Cross-Validation AUC-ROC theo Tổ Hợp Siêu Tham Số', fontweight='bold')
ax.set_ylim(min(avg_metrics) * 0.98, max(avg_metrics) * 1.01)
ax.grid(axis='y', alpha=0.3)

best_idx = np.argmax(avg_metrics)
bars[best_idx].set_color(PALETTE[2])
ax.text(best_idx, avg_metrics[best_idx] + 0.001,
        f'Best\n{avg_metrics[best_idx]:.4f}',
        ha='center', va='bottom', color=PALETTE[2], fontweight='bold', fontsize=9)

for bar, val in zip(bars, avg_metrics):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.003,
            f'{val:.4f}', ha='center', va='top', fontsize=8, color='white', alpha=0.8)

plt.tight_layout()
plt.savefig('cv_scores.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## 11. So Sánh Các Mô Hình

So sánh LinearSVC với **Logistic Regression**, **Decision Tree**, và **Random Forest**.

In [ ]:
# ── Định nghĩa các mô hình so sánh ──────────────────────────────
feature_stages = [imputer, *indexers, encoder, assembler, scaler]

models_config = {
    'LinearSVC (Best)': Pipeline(stages=[
        *feature_stages,
        LinearSVC(featuresCol='features', labelCol='label',
                  weightCol='class_weight',
                  regParam=best_lsvc.getRegParam(),
                  maxIter=best_lsvc.getMaxIter())
    ]),
    'Logistic Regression': Pipeline(stages=[
        *feature_stages,
        LogisticRegression(featuresCol='features', labelCol='label',
                           weightCol='class_weight',
                           maxIter=100, regParam=0.01, elasticNetParam=0.5)
    ]),
    'Decision Tree': Pipeline(stages=[
        *feature_stages,
        DecisionTreeClassifier(featuresCol='features', labelCol='label',
                               weightCol='class_weight',
                               maxDepth=8, impurity='gini')
    ]),
    'Random Forest': Pipeline(stages=[
        *feature_stages,
        RandomForestClassifier(featuresCol='features', labelCol='label',
                               weightCol='class_weight',
                               numTrees=50, maxDepth=6, seed=42)
    ]),
}

print(f" Định nghĩa {len(models_config)} mô hình để so sánh")

In [ ]:
# ── Huấn luyện và đánh giá tất cả mô hình ───────────────────────
results = {}

for model_name, pipeline in models_config.items():
    print(f"\n [{model_name}] Đang huấn luyện...")
    t_start = time.time()
    fitted  = pipeline.fit(train_df)
    t_train = time.time() - t_start

    preds = fitted.transform(test_df)

    auc_roc  = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC').evaluate(preds)
    auc_pr   = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderPR').evaluate(preds)
    accuracy = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy').evaluate(preds)
    f1       = MulticlassClassificationEvaluator(labelCol='label', metricName='f1').evaluate(preds)
    precision= MulticlassClassificationEvaluator(labelCol='label', metricName='weightedPrecision').evaluate(preds)
    recall   = MulticlassClassificationEvaluator(labelCol='label', metricName='weightedRecall').evaluate(preds)

    results[model_name] = {
        'AUC-ROC'  : auc_roc,
        'AUC-PR'   : auc_pr,
        'Accuracy' : accuracy,
        'F1-Score' : f1,
        'Precision': precision,
        'Recall'   : recall,
        'Time (s)' : t_train,
    }
    print(f"    Xong! AUC-ROC={auc_roc:.4f} | Acc={accuracy:.4f} | F1={f1:.4f} | Time={t_train:.1f}s")

print("\n Hoàn thành huấn luyện tất cả mô hình!")

In [ ]:
# ── Bảng so sánh ─────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.round(4)
print(" Bảng so sánh các mô hình:")
results_df

In [ ]:
# ── Biểu đồ so sánh ──────────────────────────────────────────────
metrics_to_plot = ['AUC-ROC', 'Accuracy', 'F1-Score', 'Precision', 'Recall']
model_names     = list(results.keys())

x   = np.arange(len(model_names))
w   = 0.15
offsets = np.linspace(-(len(metrics_to_plot)-1)/2*w, (len(metrics_to_plot)-1)/2*w, len(metrics_to_plot))

fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#0f1117')

for i, (metric, offset, color) in enumerate(zip(metrics_to_plot, offsets, PALETTE)):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(x + offset, vals, width=w, label=metric,
                  color=color, alpha=0.85, edgecolor='none')

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('Điểm số', fontsize=12)
ax.set_title('So Sánh Hiệu Suất Các Mô Hình Machine Learning', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.12)
ax.legend(loc='lower right', fontsize=9, framealpha=0.2)
ax.grid(axis='y', alpha=0.3)

# Highlight SVM
ax.axvspan(-0.5, 0.5, alpha=0.07, color=PALETTE[2], label='SVM')
ax.text(0, 1.08, '★ SVM', ha='center', fontsize=10, color=PALETTE[2], fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── Radar Chart (Spider Chart) ───────────────────────────────────
metrics_radar  = ['AUC-ROC', 'AUC-PR', 'Accuracy', 'F1-Score', 'Precision', 'Recall']
N = len(metrics_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d2e')
ax.set_title('Radar Chart — So Sánh Mô Hình', fontsize=13, fontweight='bold', pad=20)

for (model_name, metrics_val), color in zip(results.items(), PALETTE):
    vals = [metrics_val[m] for m in metrics_radar]
    vals += vals[:1]
    ax.plot(angles, vals, color=color, linewidth=2, label=model_name)
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_radar, size=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=8, color='gray')
ax.grid(color='#2d3561', alpha=0.5)
ax.spines['polar'].set_color('#2d3561')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=9, framealpha=0.2)

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── Trực quan hóa thời gian huấn luyện ───────────────────────────
times = {m: results[m]['Time (s)'] for m in model_names}

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#0f1117')

bars = ax.bar(list(times.keys()), list(times.values()),
              color=PALETTE, edgecolor='none', width=0.5)
for bar, val in zip(bars, times.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}s', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylabel('Thời gian (giây)')
ax.set_title('Thời Gian Huấn Luyện Các Mô Hình', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', labelsize=10)

plt.tight_layout()
plt.savefig('training_time.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## 12. Phân Tích Feature Importance (Random Forest)

> Random Forest cung cấp `featureImportances` — dùng để giải thích các đặc trưng quan trọng.

In [ ]:
# ── Lấy feature importance từ Random Forest ──────────────────────
rf_pipeline = models_config['Random Forest']
rf_fitted   = rf_pipeline.fit(train_df)
rf_model    = rf_fitted.stages[-1]

# Tên đặc trưng sau VectorAssembler
va_stage      = rf_fitted.stages[-3]   # VectorAssembler
feature_names = va_stage.getInputCols()

importances = rf_model.featureImportances.toArray()

# Ghép tên (số đặc trưng có thể nhiều hơn do OHE)
n = min(len(feature_names), len(importances))
fi_df = pd.DataFrame({
    'Feature': feature_names[:n],
    'Importance': importances[:n]
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0f1117')

colors_fi = plt.cm.plasma(np.linspace(0.2, 0.85, len(fi_df)))
bars = ax.barh(fi_df['Feature'], fi_df['Importance'],
               color=colors_fi, edgecolor='none')
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Top 15 Đặc Trưng Quan Trọng (Random Forest)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, fi_df['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, color='#a0aab4')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print("\n  Top 10 đặc trưng quan trọng nhất:")
print(fi_df.head(10).to_string(index=False))

---
## 13. Lưu & Tải Mô Hình

In [ ]:
# ── Lưu mô hình SVM tốt nhất ─────────────────────────────────────
MODEL_SAVE_PATH = r"D:\BigData\models\svm_best_model"

best_model.write().overwrite().save(MODEL_SAVE_PATH)
print(f" Đã lưu mô hình tại: {MODEL_SAVE_PATH}")

In [ ]:
# ── Tải lại mô hình ──────────────────────────────────────────────
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load(MODEL_SAVE_PATH)
loaded_pred  = loaded_model.transform(test_df.limit(5))

print(" Tải mô hình thành công! Kết quả dự đoán 5 dòng đầu:")
loaded_pred.select('label', 'prediction').show()

---
## 14. Tổng Kết

### Kết quả thực nghiệm

In [ ]:
# ── Bảng tổng kết đẹp ────────────────────────────────────────────
summary_df = results_df[['AUC-ROC', 'Accuracy', 'F1-Score', 'Precision', 'Recall', 'Time (s)']].copy()
summary_df.index.name = 'Mô hình'

# Highlight mô hình tốt nhất cho từng metric
best_row = summary_df[['AUC-ROC', 'Accuracy', 'F1-Score']].mean(axis=1).idxmax()

print("=" * 70)
print("      BẢNG TỔNG KẾT — PHÂN LOẠI SVM TRÊN DỮ LIỆU E-COMMERCE")
print("=" * 70)
print(summary_df.to_string())
print("=" * 70)
print(f"\n  Mô hình tốt nhất (theo AUC+Acc+F1 trung bình): {best_row}")